# Download data and extract Perch embeddings

Run once before training notebooks.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
%run colab_init.py

!pip install -q kaggle huggingface_hub onnxruntime-gpu soundfile tqdm

from birdclef.colab import mount_and_configure, set_kaggle_token
from birdclef.paths import EMBEDDINGS_ARCHIVE_DIR, EMBEDDINGS_TTA_ARCHIVE_DIR

mount_and_configure()
set_kaggle_token()



## Download BirdCLEF competition data

In [ ]:
import shutil
import subprocess
from pathlib import Path

from birdclef.paths import METADATA_DIR

RAW = Path("/content/bird_data")
RAW.mkdir(parents=True, exist_ok=True)

subprocess.run(
    ["kaggle", "competitions", "download", "-c", "birdclef-2026", "-p", str(RAW)],
    check=True,
)
zip_path = RAW / "birdclef-2026.zip"
if zip_path.exists():
    subprocess.run(["unzip", "-q", "-o", str(zip_path), "-d", str(RAW)], check=True)

METADATA_DIR.mkdir(parents=True, exist_ok=True)
for name in ("train.csv", "sample_submission.csv"):
    src = RAW / name
    if src.exists():
        shutil.copy(src, Path("/content") / name)
        shutil.copy(src, METADATA_DIR / name)
        print(f"Staged {name}")


## Download Perch v2 ONNX (Hugging Face)

Community ONNX export of [Google Perch v2](https://www.kaggle.com/models/google/bird-vocalization-classifier/) from [justinchuby/Perch-onnx](https://huggingface.co/justinchuby/Perch-onnx) (~394 MB). Downloaded once and cached on Drive.


In [ ]:
from birdclef.colab import ensure_perch_onnx

ensure_perch_onnx()


## Baseline embeddings (5 s windows)


In [ ]:
import pandas as pd
from birdclef.extract import extract_baseline_embeddings
from birdclef.paths import EMBEDDINGS_ARCHIVE_DIR, PERCH_ONNX, TRAIN_AUDIO_DIR, TRAIN_CSV

save_dir = str(EMBEDDINGS_ARCHIVE_DIR)

df = pd.read_csv(TRAIN_CSV)
extract_baseline_embeddings(df, str(PERCH_ONNX), str(TRAIN_AUDIO_DIR), save_dir)
print(f"Baseline embeddings saved to {save_dir}")



## TTA embeddings (2.5 s stride)


In [ ]:
from birdclef.extract import extract_tta_embeddings
from birdclef.paths import EMBEDDINGS_TTA_ARCHIVE_DIR, PERCH_ONNX, TRAIN_AUDIO_DIR

save_dir = str(EMBEDDINGS_TTA_ARCHIVE_DIR)

extract_tta_embeddings(df, str(PERCH_ONNX), str(TRAIN_AUDIO_DIR), save_dir)
print(f"TTA embeddings saved to {save_dir}")



## Optional: archive embeddings to Drive

Skip if archives already exist. Re-run after a fresh extraction to rebuild zips.


In [ ]:
import shutil
from pathlib import Path

from birdclef.paths import EMBEDDINGS_ARCHIVE_DIR, EMBEDDINGS_TTA_ARCHIVE_DIR, PROJECT_ROOT

for folder, archive_name in (
    (EMBEDDINGS_ARCHIVE_DIR, "embeddings_v2_archive.zip"),
    (EMBEDDINGS_TTA_ARCHIVE_DIR, "embeddings_v2_TTA_archive.zip"),
):
    local_zip = Path("/content") / archive_name.replace(".zip", "")
    shutil.make_archive(str(local_zip), "zip", str(folder))
    shutil.move(str(local_zip) + ".zip", str(PROJECT_ROOT / archive_name))
    print(f"Archived {folder} to {PROJECT_ROOT / archive_name}")
